
# M03 — Modify Python Before Learning Python

**Objective:** gain Python fluency by modifying and debugging a working program before studying Python systematically.

Whole-first loop:

**run → predict → modify → observe → trace → explain**



## 1. Run a complete working program first

**Predict before running:** What totals do you expect for the three orders? Which orders should receive free shipping? Which should receive a member discount?


In [ ]:

FREE_SHIPPING_THRESHOLD = 500
SHIPPING_FEE = 60
MEMBER_DISCOUNT = 0.10

orders = [
    {"id": "A101", "subtotal": 620, "items": 3, "member": True},
    {"id": "B202", "subtotal": 420, "items": 2, "member": False},
    {"id": "C303", "subtotal": 900, "items": 5, "member": True},
]

def final_total(order):
    discount = order["subtotal"] * MEMBER_DISCOUNT if order["member"] else 0
    shipping = 0 if order["subtotal"] >= FREE_SHIPPING_THRESHOLD else SHIPPING_FEE
    return round(order["subtotal"] - discount + shipping, 2)

def summarize_orders(rows):
    return [{"id": order["id"], "total": final_total(order)} for order in rows]

baseline = summarize_orders(orders)
print(baseline)

assert baseline == [
    {"id": "A101", "total": 558.0},
    {"id": "B202", "total": 480},
    {"id": "C303", "total": 810.0},
]



**What changed in your mental model after running it?**

Identify the inputs, constants, function calls, condition decisions and returned values. Use Python terminology only after locating those things in the working system.



## 2. Trace execution

**Predict before running:** For order `A101`, what values should `discount` and `shipping` take?


In [ ]:

def trace_final_total(order):
    subtotal = order["subtotal"]
    member = order["member"]
    discount = subtotal * MEMBER_DISCOUNT if member else 0
    shipping = 0 if subtotal >= FREE_SHIPPING_THRESHOLD else SHIPPING_FEE
    result = round(subtotal - discount + shipping, 2)

    trace = {
        "subtotal": subtotal,
        "member": member,
        "discount": discount,
        "shipping": shipping,
        "result": result,
    }
    print(trace)
    return result

assert trace_final_total(orders[0]) == final_total(orders[0])



## 3. Modify values and inputs

**Predict before running:** If `B202` changes from 420 to 540, what changes besides the subtotal?


In [ ]:

modified_orders = [dict(order) for order in orders]
modified_orders[1]["subtotal"] = 540

before = final_total(orders[1])
after = final_total(modified_orders[1])

print("before:", before)
print("after: ", after)

assert before == 480
assert after == 540



## 4. Modify a condition

A condition controls whether shipping is charged.

**Predict before running:** What should happen exactly at the free-shipping threshold?


In [ ]:

def shipping_fee(subtotal, threshold=FREE_SHIPPING_THRESHOLD):
    if subtotal >= threshold:
        return 0
    return SHIPPING_FEE

assert shipping_fee(499) == 60
assert shipping_fee(500) == 0
assert shipping_fee(501) == 0

print([shipping_fee(value) for value in [499, 500, 501]])



## 5. Modify a loop

**Predict before running:** Which order IDs will be collected if the loop keeps only totals of at least 600?


In [ ]:

large_order_ids = []

for order in orders:
    total = final_total(order)
    if total >= 600:
        large_order_ids.append(order["id"])

print(large_order_ids)
assert large_order_ids == ["C303"]



## 6. Modify a function

Functions let us change one part of behavior while keeping the calling code stable.

**Predict before running:** What happens to a member order if the discount parameter changes from 10% to 5%?


In [ ]:

def configurable_total(
    order,
    member_discount=MEMBER_DISCOUNT,
    free_shipping_threshold=FREE_SHIPPING_THRESHOLD,
):
    discount = order["subtotal"] * member_discount if order["member"] else 0
    shipping = 0 if order["subtotal"] >= free_shipping_threshold else SHIPPING_FEE
    return round(order["subtotal"] - discount + shipping, 2)

ten_percent = configurable_total(orders[0], member_discount=0.10)
five_percent = configurable_total(orders[0], member_discount=0.05)

print("10%:", ten_percent)
print("5%: ", five_percent)

assert five_percent > ten_percent



## 7. Modify lists and dictionaries

Each order is a dictionary. `orders` is a list of those dictionaries.

**Predict before running:** After copying the list and adding a `priority` key, should the original dictionaries change?


In [ ]:

enriched_orders = [dict(order) for order in orders]

for order in enriched_orders:
    order["priority"] = final_total(order) >= 600

print(enriched_orders)

assert "priority" in enriched_orders[0]
assert "priority" not in orders[0]



## 8. Debug runtime failures

Do not edit randomly. Read the exception type, identify the failing operation, form a hypothesis, then make the smallest repair.

The following examples intentionally trigger failures but catch them so Restart + Run All remains clean.


In [ ]:

def observe_exception(label, operation):
    try:
        operation()
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
        return type(exc).__name__
    return "NO_ERROR"

name_error = observe_exception(
    "missing variable",
    lambda: eval("missing_total + 1"),
)

type_error = observe_exception(
    "wrong operand type",
    lambda: "420" + 10,
)

key_error = observe_exception(
    "missing dictionary key",
    lambda: orders[0]["customer_name"],
)

assert name_error == "NameError"
assert type_error == "TypeError"
assert key_error == "KeyError"



## 9. Debug a condition-direction bug

This version runs without an exception but makes the wrong decision.

**Predict before running:** Which boundary cases will expose the mistake most clearly?


In [ ]:

def shipping_fee_broken(subtotal):
    if subtotal <= FREE_SHIPPING_THRESHOLD:
        return 0
    return SHIPPING_FEE

cases = [420, 500, 620]

for value in cases:
    print(
        value,
        "broken=", shipping_fee_broken(value),
        "expected=", shipping_fee(value),
    )

assert shipping_fee_broken(420) != shipping_fee(420)
assert shipping_fee_broken(620) != shipping_fee(620)



## 10. Debug an off-by-one failure

**Predict before running:** A list with three elements has which valid index values?


In [ ]:

def last_id_broken(rows):
    return rows[len(rows)]["id"]

off_by_one = observe_exception(
    "off by one",
    lambda: last_id_broken(orders),
)

def last_id_fixed(rows):
    return rows[len(rows) - 1]["id"]

assert off_by_one == "IndexError"
assert last_id_fixed(orders) == "C303"
print(last_id_fixed(orders))



## 11. Controlled failure

The next function produces a plausible number but contains **one narrow seeded root cause**.

Before reading further:

1. record expected and observed output;
2. identify useful intermediate values;
3. state one hypothesis;
4. identify one smallest test that could falsify it.


In [ ]:

def buggy_final_total(order):
    discount = order["subtotal"] * MEMBER_DISCOUNT if order["member"] else 0
    shipping = 0 if order["subtotal"] >= FREE_SHIPPING_THRESHOLD else SHIPPING_FEE
    return round(order["subtotal"] + discount + shipping, 2)

failing_order = orders[0]
expected = final_total(failing_order)
observed = buggy_final_total(failing_order)

print("expected:", expected)
print("observed:", observed)
assert observed != expected


In [ ]:

def trace_buggy_total(order):
    subtotal = order["subtotal"]
    discount = subtotal * MEMBER_DISCOUNT if order["member"] else 0
    shipping = 0 if subtotal >= FREE_SHIPPING_THRESHOLD else SHIPPING_FEE

    trace = {
        "subtotal": subtotal,
        "discount": discount,
        "shipping": shipping,
        "observed": round(subtotal + discount + shipping, 2),
        "expected": final_total(order),
    }
    return trace

failure_trace = trace_buggy_total(failing_order)
print(failure_trace)



**Pause before the repair.**

Write down:

- the symptom;
- your hypothesis;
- the first incorrect operation or intermediate value;
- the smallest repair;
- the cases you will rerun after repairing it.


In [ ]:

def repaired_final_total(order):
    discount = order["subtotal"] * MEMBER_DISCOUNT if order["member"] else 0
    shipping = 0 if order["subtotal"] >= FREE_SHIPPING_THRESHOLD else SHIPPING_FEE
    return round(order["subtotal"] - discount + shipping, 2)

for order in orders:
    assert repaired_final_total(order) == final_total(order)

boundary_cases = [
    {"id": "X1", "subtotal": 499, "items": 1, "member": False},
    {"id": "X2", "subtotal": 500, "items": 1, "member": False},
    {"id": "X3", "subtotal": 500, "items": 1, "member": True},
]

for order in boundary_cases:
    print(order["id"], repaired_final_total(order))



## 12. Code reading before modification

Do not run the next cell immediately.

Trace it manually and predict the exact dictionary returned for the existing `orders`.


In [ ]:

def bucket_order_ids(rows):
    buckets = {"small": [], "large": []}

    for order in rows:
        bucket = "large" if final_total(order) >= 600 else "small"
        buckets[bucket].append(order["id"])

    return buckets

bucketed = bucket_order_ids(orders)
print(bucketed)

assert bucketed == {
    "small": ["A101", "B202"],
    "large": ["C303"],
}



## 13. Explain, don't merely produce output

For at least three modifications answer:

- What did you predict?
- What line did you change?
- What changed in execution?
- What output changed?
- What remained unchanged?
- What Python concept explains the behavior?
- What evidence supports your explanation?



## 14. No-AI Gate

Complete `missions/M03/no_ai_gate.md` on the fresh inventory program **without AI-generated code**.

The gate checks transfer rather than memory of this notebook.



## 15. Assessment and evidence

Submit evidence required by `missions/M03/evidence_contract.yaml`.

Passing M03 means you can:

- predict behavior before running;
- make targeted changes to unfamiliar Python;
- trace variables and control flow;
- distinguish runtime failures from logic failures;
- repair a failure from evidence;
- explain why the repair works;
- transfer those skills to fresh code without AI-generated code.

M03 is not a syntax-recall test.
